In [18]:
import feedparser 
url = "https://www.cert.ssi.gouv.fr/feed/" 
rss_feed = feedparser.parse(url) 
entries_link = []
for entry in rss_feed.entries: 
    print("Titre :", entry.title) 
    print("Description:", entry.description) 
    print("Lien :", entry.link) 
    entries_link.append(entry.link)
    print("Date :", entry.published)

Titre : Multiples vulnérabilités dans Microsoft Azure Linux (05 juin 2026)
Description: De multiples vulnérabilités ont été découvertes dans Microsoft Azure Linux. Elles permettent à un attaquant de provoquer un problème de sécurité non spécifié par l'éditeur.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0693/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Vulnérabilité dans Cisco Catalyst SD-WAN (05 juin 2026)
Description: Une vulnérabilité a été découverte dans Cisco Catalyst SD-WAN. Elle permet à un attaquant de provoquer une élévation de privilèges. Cisco indique que la vulnérabilité CVE-2026-20245 est activement exploitée.
Lien : https://www.cert.ssi.gouv.fr/avis/CERTFR-2026-AVI-0699/
Date : Fri, 05 Jun 2026 00:00:00 +0000
Titre : Multiples vulnérabilités dans le noyau Linux de SUSE (05 juin 2026)
Description: De multiples vulnérabilités ont été découvertes dans le noyau Linux de SUSE. Certaines d'entre elles permettent à un attaquant de provoquer une atteinte à la con

In [23]:
import requests 
import re 
for link in entries_link:
    url = link + "json/"
    
    try:
        response = requests.get(url)
        # Lève une erreur si le code HTTP n'est pas 200 (ex: 404, 500)
        response.raise_for_status() 
        
        # Tente de décoder le JSON
        data = response.json() 
        
    except requests.exceptions.RequestException as e:
        # Attrape les erreurs de connexion, timeout, 404, etc.
        print(f"Erreur de connexion ou HTTP pour {url} : {e}")
        continue # Passe au lien suivant
        
    except requests.exceptions.JSONDecodeError:
        # Attrape le problème que tu avais (réponse non-JSON)
        print(f"Réponse JSON invalide pour {url}. Le site a renvoyé du texte brut ou du HTML.")
        # Optionnel : décommenter la ligne suivante pour inspecter le problème en direct
        # print("Contenu reçu :", response.text[:200]) 
        continue # Passe au lien suivant

    # --- Tout ce qui est en dessous ne s'exécute que si le try a réussi ---
    
    # Extraction des CVE références dans la clé cves du dict data 
    # (Sécurisé avec .get() au cas où la clé "cves" serait absente)
    cves_data = data.get("cves", [])
    ref_cves = list(cves_data)  
    print("CVE référencés :", ref_cves) 
    
    # Extraction des CVE avec une regex 
    cve_pattern = r"CVE-\d{4}-\d{4,7}" 
    cve_list = list(set(re.findall(cve_pattern, str(data)))) 
    print("CVE trouvés :", cve_list)
    print("-" * 40) # Petite ligne de séparation pour y voir plus clair dans ta console

CVE référencés : [{'name': 'CVE-2026-42250', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-42250'}, {'name': 'CVE-2026-9149', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-9149'}, {'name': 'CVE-2026-9150', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-9150'}]
CVE trouvés : ['CVE-2026-42250', 'CVE-2026-9150', 'CVE-2026-9149']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-20245', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-20245'}]
CVE trouvés : ['CVE-2026-20245']
----------------------------------------
CVE référencés : [{'name': 'CVE-2026-43366', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-43366'}, {'name': 'CVE-2026-23260', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23260'}, {'name': 'CVE-2026-23447', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23447'}, {'name': 'CVE-2026-23387', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-23387'}, {'name': 'CVE-2026-31658', 'url': 'https://www.cve.org/CVERecord?id=CVE-2026-316

In [21]:
import requests 
cve_id = "CVE-2023-24488" 
url = f"https://cveawg.mitre.org/api/cve/{cve_id}" 
response = requests.get(url) 
data = response.json() 
# Extraire la description  
description = data["containers"]["cna"]["descriptions"][0]["value"]  
# Extraire le score CVSS  
#ATTENTION tous les CVE ne contiennent pas nécessairement ce champ, gérez l’exception,  
#ou peut etre au lieu de cvssV3_0 c’est cvssV3_1 ou autre clé 
cvss_score =data["containers"]["cna"]["metrics"][0]["cvssV3_1"]["baseScore"]  
cwe = "Non disponible" 
cwe_desc="Non disponible" 
problemtype = data["containers"]["cna"].get("problemTypes", {}) 
if problemtype and "descriptions" in problemtype[0]: 
    cwe = problemtype[0]["descriptions"][0].get("cweId", "Non disponible") 
    cwe_desc=problemtype[0]["descriptions"][0].get("description", "Non disponible") 
# Extraire les produits affectés  
affected = data["containers"]["cna"]["affected"] 
for product in affected: 
    vendor = product["vendor"] 
    product_name = product["product"]  
    versions = [v["version"] for v in product["versions"] if v["status"] == "affected"] 
    print(f"Éditeur : {vendor}, Produit : {product_name}, Versions : {', '.join(versions)}") 
# Afficher les résultats  
print(f"CVE : {cve_id}")  
print(f"Description : {description}")  
print(f"Score CVSS : {cvss_score}") 
print(f"Type CWE : {cwe}") 
print(f"CWE Description : {cwe_desc}") 

Éditeur : Citrix, Produit : Citrix ADC and Citrix Gateway , Versions : 13.1, 13.0, 12.1, 12.1-FIPS , 13.1-FIPS , 12.1-NDcPP
CVE : CVE-2023-24488
Description : Cross site scripting vulnerability in Citrix ADC and Citrix Gateway  in allows and attacker to perform cross site scripting
Score CVSS : 6.1
Type CWE : CWE-79
CWE Description : CWE-79 Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')


In [22]:
import requests 
# URL de l'API EPSS pour récupérer la probabilité d'exploitation 
cve_id = "CVE-2023-46805" 
url = f"https://api.first.org/data/v1/epss?cve={cve_id}" 
# Requête GET pour récupérer les données JSON 
response = requests.get(url) 
data = response.json() 
# Extraire le score EPSS 
epss_data = data.get("data", []) 
if epss_data: 
    epss_score = epss_data[0]["epss"]
    print(f"CVE : {cve_id}") 
    print(f"Score EPSS : {epss_score}") 
else: 
    print(f"Aucun score EPSS trouvé pour {cve_id}") 

CVE : CVE-2023-46805
Score EPSS : 0.943670000
